# NB-701: Building Agentic Systems

Create AI agents with tools and memory.

## Learning Objectives
- Define agent architecture
- Implement tools for agents
- Add memory to agents
- Build multi-agent systems

In [ ]:
import json
import time
from typing import List, Dict, Optional

print("Libraries imported")

## 1. Define Tools

In [ ]:
class Tool:
    def __init__(self, name, description, func):
        self.name = name
        self.description = description
        self.func = func

    def run(self, **kwargs):
        return self.func(**kwargs)

# TODO: Define some tools
def calculator(expression: str) -> str:
    """Evaluate a math expression."""
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

def get_weather(location: str) -> str:
    """Get weather for a location."""
    # Simulated
    return f"Weather in {location}: 22°C, Sunny"

def search(query: str) -> str:
    """Search for information."""
    return f"Search results for '{query}': [Simulated results]"

tools = [
    Tool("calculator", "Evaluate mathematical expressions", calculator),
    Tool("weather", "Get current weather", get_weather),
    Tool("search", "Search the web", search),
]

print(f"Available tools: {[t.name for t in tools]}")

## 2. Simple Agent with Tools

In [ ]:
class SimpleAgent:
    def __init__(self, name: str, tools: List[Tool]):
        self.name = name
        self.tools = {tool.name: tool for tool in tools}
        self.memory = []

    def think(self, query: str) -> Dict:
        """Decide which tool to use."""
        # TODO: Simple keyword-based decision
        query_lower = query.lower()

        # Check for calculator
        if any(word in query_lower for word in ["calculate", "compute", "math", "+", "-", "*", "/"]):
            return {"tool": "calculator", "input": query}

        # Check for weather
        if "weather" in query_lower:
            return {"tool": "weather", "input": query}

        # Default to search
        return {"tool": "search", "input": query}

    def act(self, tool_name: str, tool_input: str) -> str:
        """Execute a tool."""
        if tool_name in self.tools:
            return self.tools[tool_name].run(input=tool_input)
        return "Tool not found"

    def run(self, query: str) -> Dict:
        """Full agent loop."""
        # Think
        decision = self.think(query)

        # Act
        result = self.act(decision["tool"], decision["input"])

        # Remember
        self.memory.append({
            "query": query,
            "tool": decision["tool"],
            "result": result,
            "timestamp": time.time()
        })

        return {
            "tool_used": decision["tool"],
            "result": result
        }

# Create agent
agent = SimpleAgent("Agent1", tools)
print(f"Agent '{agent.name}' created with {len(agent.tools)} tools")

## 3. Test Agent

In [ ]:
# TODO: Test with different queries
queries = [
    "What's 25 * 34?",
    "What's the weather in Paris?",
    "Search for AI news",
]

for query in queries:
    print(f"\nQuery: {query}")
    response = agent.run(query)
    print(f"Tool: {response['tool_used']}")
    print(f"Result: {response['result']}")

## 4. Agent with Memory

In [ ]:
class AgentWithMemory(SimpleAgent):
    def __init__(self, name: str, tools: List[Tool], memory_size: int = 10):
        super().__init__(name, tools)
        self.memory_size = memory_size

    def remember(self, key: str, value: str):
        """Store information in memory."""
        self.memory.append({"key": key, "value": value, "timestamp": time.time()})

        # Limit memory size
        if len(self.memory) > self.memory_size:
            self.memory = self.memory[-self.memory_size:]

    def recall(self, key: str) -> Optional[str]:
        """Recall information from memory."""
        for mem in reversed(self.memory):
            if "key" in mem and mem["key"] == key:
                return mem["value"]
        return None

    def get_memory_summary(self) -> str:
        """Get summary of memory."""
        items = [f"{mem.get('key', 'unknown')}: {mem.get('value', 'unknown')}" for mem in self.memory]
        return "\n".join(items)

# Create agent with memory
memory_agent = AgentWithMemory("MemoryAgent", tools)

# Store and recall
memory_agent.remember("user_name", "Alice")
memory_agent.remember("user_location", "Paris")

print(f"Recall user_name: {memory_agent.recall('user_name')}")
print(f"\nMemory summary:\n{memory_agent.get_memory_summary()}")

## 5. Multi-Agent System

In [ ]:
class MultiAgentSystem:
    def __init__(self):
        self.agents = {}

    def add_agent(self, agent: SimpleAgent):
        self.agents[agent.name] = agent

    def delegate(self, task: str, agent_name: str = None) -> Dict:
        """Delegate task to specific or best agent."""

        if agent_name and agent_name in self.agents:
            return self.agents[agent_name].run(task)

        # Auto-select agent (simple round-robin)
        agent_names = list(self.agents.keys())
        selected = agent_names[0]  # Always select first for now

        return {
            "agent": selected,
            "result": self.agents[selected].run(task)
        }

# Create multi-agent system
multi_agent = MultiAgentSystem()
multi_agent.add_agent(SimpleAgent("Researcher", [tools[0], tools[2]]))  # Calculator, Search
multi_agent.add_agent(SimpleAgent("Assistant", [tools[1], tools[2]]))  # Weather, Search

print(f"Agents in system: {list(multi_agent.agents.keys())}")

# Delegate task
result = multi_agent.delegate("Calculate 15 * 23")
print(f"\nDelegated to: {result['agent']}")
print(f"Result: {result['result']}")

## Exercise

1. Add more tools to the agent
2. Implement better tool selection logic
3. Add agent communication
4. Implement a supervisor agent